# Notebook 35 — Scheduling & Triggers

Run agents on a schedule or in response to events.

| Class | Fires when |
|---|---|
| `IntervalSchedule` | every N seconds/minutes/hours |
| `CronSchedule` | matching minute/hour/day/weekday |
| `OnceSchedule` | exactly once at a set time |
| `Trigger` | named event emitted via `scheduler.emit()` |
| `Scheduler` | async loop orchestrating all jobs |

## 1. Setup

In [ ]:
import asyncio, time
from multigen.scheduler import (
    Scheduler, ScheduledJob, IntervalSchedule, CronSchedule,
    OnceSchedule, Trigger, JobResult,
)
print('Scheduler module loaded')

## 2. Interval-scheduled jobs

In [ ]:
results = []
scheduler = Scheduler(tick_interval=0.05)

@scheduler.job(IntervalSchedule(seconds=0.1), name='heartbeat')
async def heartbeat():
    results.append(f'beat@{time.time():.2f}')
    return 'ok'

@scheduler.job(IntervalSchedule(minutes=0.002), name='metrics')  # 0.12 s
async def collect_metrics():
    results.append('metrics')
    return {'cpu': 42, 'mem': 1024}

await scheduler.start()
await asyncio.sleep(0.4)
await scheduler.stop()

beats  = [r for r in results if 'beat' in r]
metric = [r for r in results if r == 'metrics']
print(f'heartbeat fired {len(beats)}x')
print(f'metrics fired {len(metric)}x')

## 3. Event triggers

In [ ]:
events = []
scheduler2 = Scheduler(tick_interval=0.05)

# Trigger on any 'order.*' event
order_trigger = Trigger('order.placed')
@scheduler2.on(order_trigger, name='process-order')
async def process_order(data):
    events.append(f'order:{data["id"]}')
    return {'status': 'processed', 'order_id': data['id']}

# Filtered trigger — only high-value orders
high_value = Trigger('order.placed', filter_fn=lambda d: d.get('total', 0) > 100)
@scheduler2.on(high_value, name='priority-fulfillment')
async def priority_fulfillment(data):
    events.append(f'priority:{data["id"]}')
    return {'status': 'expedited'}

await scheduler2.start()

# Emit events
n1 = await scheduler2.emit('order.placed', {'id': 'A1', 'total': 50})
n2 = await scheduler2.emit('order.placed', {'id': 'A2', 'total': 200})
print(f'Event A1 triggered {n1} jobs')
print(f'Event A2 triggered {n2} jobs')

await asyncio.sleep(0.2)
await scheduler2.stop()
print('Events processed:', events)

## 4. Run-once job

In [ ]:
ran_once = []
scheduler3 = Scheduler(tick_interval=0.05)

@scheduler3.job(OnceSchedule(delay_s=0.1), name='startup-init')
async def startup_init():
    ran_once.append(time.time())
    return 'initialised'

await scheduler3.start()
await asyncio.sleep(0.4)
await scheduler3.stop()

print(f'startup-init ran {len(ran_once)}x (should be 1)')

## 5. Manual trigger and error handling

In [ ]:
scheduler4 = Scheduler()
errors = []

async def on_error(exc, result):
    errors.append(str(exc))

scheduler4.add_job(ScheduledJob(
    fn=lambda: (_ for _ in ()).throw(ValueError('intentional error')),
    name='flaky-job',
    schedule=IntervalSchedule(hours=999),
    on_error=on_error,
))

result = await scheduler4.run_now('flaky-job')
print(f'Flaky job: success={result.success}, error={result.error}')

## 6. Stats

In [ ]:
stats = scheduler.stats()
print('Scheduler stats:')
for job_name, job_stats in stats['jobs'].items():
    print(f'  {job_name}: runs={job_stats["run_count"]} errors={job_stats["error_count"]}')

## Summary

```python
# Typical production pattern
scheduler = Scheduler(tick_interval=1.0)

@scheduler.job(CronSchedule(hour=2, minute=0), name='nightly-report')
async def nightly_report():
    ...

@scheduler.on(Trigger('payment.completed'), name='send-receipt')
async def send_receipt(data):
    ...

await scheduler.start()
# Keep running indefinitely
```